# Day 014 | Regular Expressions (re module)

## 100 Days of ML for Geotechnical Engineering
**Phase:** Python Fundamentals

**Author:** Ripon Chandra Malo  
**Date:** YYYY-MM-DD  

---

## What We Will Learn Today

| Concept | What It Means | Geotechnical Example |
|---------|--------------|---------------------|
| `re.search()` | Find a pattern anywhere in text | Find depth value in `"BH-01 @ 3.5m"` |
| `re.findall()` | Find ALL matches in text | Extract every N-value from a log |
| `re.match()` | Match at the START of text | Validate borehole ID format |
| `re.sub()` | Find and replace with patterns | Clean messy field notes |
| `\d`, `\w`, `\s` | Digit, word char, whitespace | Match numbers, soil codes, spacing |
| `+`, `*`, `?` | Quantifiers (1+, 0+, 0 or 1) | Match varying digit counts |
| `()` Groups | Capture specific parts | Extract depth AND N-value separately |
| `[A-Z]`, `[0-9]` | Character classes | Match USCS codes like `SP`, `CL` |
| Raw strings `r""` | Avoid backslash escaping | `r"\d+\.\d+"` instead of `"\\d+\\.\\d+"` |
| Compiled patterns | Reusable, efficient regex | Parse 1000 borehole entries fast |

---

## Our Geotechnical Problem

Geotechnical data often arrives as **messy, unstructured text**:

```
BH-01 @ 3.5m: N=24, Silty Sand (SM), w=18.5%
BH-01 @ 6.0m: N=12, Clay (CL), w=32.1%
Borehole 02, depth 9.0 m — SPT blow count 35, Gravel (GP)
GPS: 40.7608N, 111.8910W  elev. 1320.5m
```

This text contains valuable data — depths, N-values, soil types, classifications, coordinates — but it is trapped in free-form strings. You cannot simply split by commas because the format is inconsistent.

**Regular expressions** let us write *patterns* that describe what the data looks like, then extract exactly the fields we need — no matter how messy the text is.

---

## Part 1: Why Regex? — The String Methods Wall

Python string methods (`split`, `find`, `replace`) work for simple cases. But real geotechnical text breaks them fast:

```python
# Simple: split works
"3.5,24,SM".split(",")  # ['3.5', '24', 'SM']

# Real data: split fails
"BH-01 @ 3.5m: N=24, Silty Sand (SM), w=18.5%"  # Split on what?
```

Regex lets you say: *"find a number followed by 'm'"* or *"find two uppercase letters inside parentheses"* — patterns that adapt to messy formatting.

In [ ]:
# ─────────────────────────────────────────────────────
# WHY REGEX? — String methods vs regular expressions
# ─────────────────────────────────────────────────────

import re

# A real borehole log entry
entry = "BH-01 @ 3.5m: N=24, Silty Sand (SM), w=18.5%"

# Attempt 1: string methods (fragile, verbose)
# To get depth: find '@', then find 'm', slice between...
at_pos = entry.find("@")
m_pos = entry.find("m", at_pos)
depth_str = entry[at_pos+1:m_pos].strip()
print(f"String method depth: '{depth_str}'")
# This breaks if format changes even slightly!
print()

# Attempt 2: regex (robust, concise)
depth = re.search(r"(\d+\.\d+)m", entry)
n_val = re.search(r"N=(\d+)", entry)
uscs = re.search(r"\(([A-Z]{2})\)", entry)

print(f"Regex depth: {depth.group(1)}")
print(f"Regex N:     {n_val.group(1)}")
print(f"Regex USCS:  {uscs.group(1)}")
print()
print("Three clean extractions, each in one line.")
print("And they work even if spacing or wording changes!")

---

## Part 2: Basic Patterns — The Building Blocks

Regex patterns are built from simple pieces:

| Pattern | Matches | Example Match |
|---------|---------|---------------|
| `\d` | Any digit (0-9) | `3`, `7`, `0` |
| `\d+` | One or more digits | `24`, `135`, `7` |
| `\d*` | Zero or more digits | `""`, `5`, `42` |
| `\w` | Word character (letter, digit, `_`) | `a`, `B`, `3`, `_` |
| `\s` | Whitespace (space, tab, newline) | ` `, `\t` |
| `.` | Any character (except newline) | `a`, `3`, `@` |
| `\.` | A literal dot | `.` |
| `+` | 1 or more of previous | `\d+` = one or more digits |
| `*` | 0 or more of previous | `\d*` = zero or more digits |
| `?` | 0 or 1 of previous | `\d?` = optional digit |

**Raw strings** (`r"..."`) are essential: `r"\d+"` means the regex `\d+`, while `"\d+"` means Python tries to interpret `\d` as an escape sequence.

**Rule:** Always use `r"..."` for regex patterns.

In [ ]:
# ─────────────────────────────────────────────────────
# BASIC PATTERNS — Matching digits, words, and dots
# ─────────────────────────────────────────────────────

import re

text = "BH-01 @ 3.5m: N=24, Silty Sand (SM), w=18.5%"

# \d+ : one or more digits
digits = re.findall(r"\d+", text)
print(f"All digit groups: {digits}")
# Finds: 01, 3, 5, 24, 18, 5

# \d+\.\d+ : decimal numbers (digits.digits)
decimals = re.findall(r"\d+\.\d+", text)
print(f"Decimal numbers: {decimals}")
# Finds: 3.5, 18.5

# \w+ : word characters (letters, digits, underscore)
words = re.findall(r"\w+", text)
print(f"All words: {words}")

# [A-Z]{2} : exactly two uppercase letters
codes = re.findall(r"[A-Z]{2}", text)
print(f"Two-letter codes: {codes}")
print()

# Combine patterns for geotechnical meaning
print("Meaningful patterns:")
print(f"  Depths (num + m):  {re.findall(r'\d+\.\d+m', text)}")
print(f"  N-values (N=num):  {re.findall(r'N=\d+', text)}")
print(f"  Moisture (num%):   {re.findall(r'\d+\.\d+%', text)}")

---

## Part 3: Core Functions — search, findall, match, sub

The `re` module has four main functions:

| Function | Purpose | Returns |
|----------|---------|--------|
| `re.search(pattern, text)` | Find FIRST match anywhere | Match object or `None` |
| `re.findall(pattern, text)` | Find ALL matches | List of strings |
| `re.match(pattern, text)` | Match at START only | Match object or `None` |
| `re.sub(pattern, repl, text)` | Find and replace | New string |

A **match object** has:
- `.group()` — the full matched text
- `.group(1)` — first captured group (inside parentheses)
- `.start()`, `.end()` — position in original text

In [ ]:
# ─────────────────────────────────────────────────────
# re.search() — Find the FIRST match anywhere
# ─────────────────────────────────────────────────────

entry = "BH-03 @ 6.0m: N=18, Clayey Sand (SC), w=22.3%"

# Search for depth
match = re.search(r"\d+\.\d+m", entry)
if match:
    print(f"Found depth: '{match.group()}' at position {match.start()}-{match.end()}")

# Search for N-value
match = re.search(r"N=(\d+)", entry)
if match:
    print(f"Full match: '{match.group()}'")
    print(f"N-value:    '{match.group(1)}'")

# When search finds nothing — returns None
match = re.search(r"RQD=\d+", entry)
print(f"RQD search: {match}")  # None
print()

# SAFE pattern: always check for None before .group()
match = re.search(r"RQD=(\d+)", entry)
rqd = match.group(1) if match else "not recorded"
print(f"RQD: {rqd}")

In [ ]:
# ─────────────────────────────────────────────────────
# re.findall() — Find ALL matches
# ─────────────────────────────────────────────────────

# A multi-line borehole log
log = """BH-01 @ 1.5m: N=5, Fill
BH-01 @ 3.0m: N=8, Silty Sand (SM)
BH-01 @ 4.5m: N=12, Sand (SP)
BH-01 @ 6.0m: N=18, Clayey Sand (SC)
BH-01 @ 7.5m: N=28, Sand (SW)
BH-01 @ 9.0m: N=35, Gravel (GP)
BH-01 @ 10.5m: N=50, Weathered Rock
BH-01 @ 12.0m: N=60, Bedrock"""

# Extract all depths
depths = re.findall(r"(\d+\.\d+)m", log)
print(f"Depths:  {depths}")

# Extract all N-values
n_values = re.findall(r"N=(\d+)", log)
print(f"N-vals:  {n_values}")

# Extract all USCS codes (two uppercase letters in parentheses)
uscs = re.findall(r"\(([A-Z]{2})\)", log)
print(f"USCS:    {uscs}")

# Extract all borehole IDs
bh_ids = re.findall(r"BH-\d+", log)
print(f"BH IDs:  {list(set(bh_ids))}")
print()

# Convert to numbers for analysis
depths_num = [float(d) for d in depths]
n_num = [int(n) for n in n_values]
print(f"Depths (float): {depths_num}")
print(f"N (int):        {n_num}")
print(f"Avg N:          {sum(n_num)/len(n_num):.1f}")

In [ ]:
# ─────────────────────────────────────────────────────
# re.match() — Match at the START of the string
# re.sub()   — Find and REPLACE
# ─────────────────────────────────────────────────────

# re.match() — only checks the BEGINNING
# Useful for validating borehole ID format
test_ids = ["BH-01", "BH-12", "CPT-03", "bh-05", "HOLE7"]

print("Validating borehole ID format (BH-##):")
for test_id in test_ids:
    match = re.match(r"BH-\d{2}$", test_id)
    status = "valid" if match else "INVALID"
    print(f"  {test_id:8s} -> {status}")
print()

# re.sub() — clean up messy field notes
messy_note = "depth:  3.5 m ,  N =  24  , soil:  SILTY   SAND"
print(f"Messy:   '{messy_note}'")

# Remove extra spaces (replace 2+ spaces with 1)
cleaned = re.sub(r"\s{2,}", " ", messy_note)
print(f"Cleaned: '{cleaned}'")

# Remove spaces around = and :
cleaned = re.sub(r"\s*=\s*", "=", cleaned)
cleaned = re.sub(r"\s*:\s*", ":", cleaned)
print(f"Tight:   '{cleaned}'")
print()

# Replace non-standard depth formats
notes = "Depth 3.5 meters, depth 6.0 m, at 9.0m depth"
standardized = re.sub(r"(\d+\.\d+)\s*(?:meters|m)", r"\1m", notes)
print(f"Original:     '{notes}'")
print(f"Standardized: '{standardized}'")

---

## Part 4: Groups and Capturing — Extract Multiple Fields at Once

Parentheses `()` create **capturing groups** — they extract specific parts of the match.

```python
# Without groups: get the whole match
re.search(r"N=\d+", text).group()   # 'N=24'

# With groups: get just the number
re.search(r"N=(\d+)", text).group(1)  # '24'

# Multiple groups: get several fields
m = re.search(r"(\d+\.\d+)m.*N=(\d+)", text)
m.group(1)  # '3.5'  (depth)
m.group(2)  # '24'   (N-value)
```

With `findall()`, groups change the output — instead of returning full matches, it returns just the grouped parts.

In [ ]:
# ─────────────────────────────────────────────────────
# GROUPS — Extract multiple fields from one pattern
# ─────────────────────────────────────────────────────

entry = "BH-01 @ 3.5m: N=24, Silty Sand (SM), w=18.5%"

# Single group — extract just the value
m = re.search(r"N=(\d+)", entry)
print(f"N-value: {m.group(1)}")

# Multiple groups — extract borehole ID, depth, N-value
m = re.search(r"(BH-\d+) @ (\d+\.\d+)m.*N=(\d+)", entry)
if m:
    print(f"BH ID:   {m.group(1)}")
    print(f"Depth:   {m.group(2)}m")
    print(f"N-value: {m.group(3)}")
    print(f"All:     {m.groups()}")  # Tuple of all groups
print()

# findall with groups — returns group contents, not full match
log = """BH-01 @ 1.5m: N=5, Fill
BH-01 @ 3.0m: N=8, Silty Sand (SM)
BH-01 @ 6.0m: N=18, Clayey Sand (SC)
BH-01 @ 9.0m: N=35, Gravel (GP)"""

# Extract (depth, N-value) pairs
pairs = re.findall(r"(\d+\.\d+)m.*?N=(\d+)", log)
print("Depth-N pairs:")
for depth, n in pairs:
    print(f"  {depth}m -> N={n}")

In [ ]:
# ─────────────────────────────────────────────────────
# NAMED GROUPS — (?P<name>...) for readable extraction
# ─────────────────────────────────────────────────────

entry = "BH-01 @ 3.5m: N=24, Silty Sand (SM), w=18.5%"

# Named groups: (?P<name>pattern)
pattern = (
    r"(?P<bh_id>BH-\d+)"
    r" @ (?P<depth>\d+\.\d+)m"
    r".*?N=(?P<n_value>\d+)"
    r".*?\((?P<uscs>[A-Z]{2})\)"
    r".*?w=(?P<moisture>\d+\.\d+)%"
)

m = re.search(pattern, entry)
if m:
    print("Named group access:")
    print(f"  BH:       {m.group('bh_id')}")
    print(f"  Depth:    {m.group('depth')}m")
    print(f"  N-value:  {m.group('n_value')}")
    print(f"  USCS:     {m.group('uscs')}")
    print(f"  Moisture: {m.group('moisture')}%")
    print()
    
    # .groupdict() returns a dictionary!
    data = m.groupdict()
    print(f"As dict: {data}")
    print()
    print("Named groups turn regex output into dictionaries.")
    print("This is how we go from text -> structured data.")

---

## Part 5: Character Classes and Ranges

Square brackets `[]` define a **character class** — match any ONE character from the set.

| Pattern | Matches | Use Case |
|---------|---------|----------|
| `[A-Z]` | Any uppercase letter | USCS codes: `S`, `C`, `G` |
| `[a-z]` | Any lowercase letter | Soil descriptions |
| `[0-9]` | Any digit (same as `\d`) | Depths, N-values |
| `[A-Za-z]` | Any letter | Mixed-case soil names |
| `[SCGM][WPLHCT]` | USCS first + second letter | `SW`, `CL`, `GM`, `PT` |
| `[^\d]` | NOT a digit | Non-numeric characters |
| `{n}` | Exactly n repetitions | `[A-Z]{2}` = two letters |

In [ ]:
# ─────────────────────────────────────────────────────
# CHARACTER CLASSES — Matching USCS codes and coordinates
# ─────────────────────────────────────────────────────

# Parse USCS codes from a site report
report = """Layers encountered: Fill, Silty Sand (SM), Poorly-graded Sand (SP),
Low-plasticity Clay (CL), High-plasticity Clay (CH), Silty Gravel (GM),
Clayey Sand (SC), and Peat (PT) at the base."""

# Match valid USCS codes in parentheses
# First letter: S(and), G(ravel), C(lay), M(silt), O(rganic), P(eat)
# Second letter: W, P, L, H, C, T, M
uscs_codes = re.findall(r"\(([SGCMOP][WPLHCTM])\)", report)
print(f"USCS codes found: {uscs_codes}")
print()

# Extract GPS coordinates
gps_text = """Site coordinates:
BH-01: 40.7608N, 111.8910W, elev. 1320.5m
BH-02: 40.7615N, 111.8895W, elev. 1318.2m
BH-03: 40.7602N, 111.8920W, elev. 1322.1m"""

# Pattern: digits.digits followed by N/S or E/W
coords = re.findall(
    r"(\d+\.\d+)([NS]),\s*(\d+\.\d+)([EW])",
    gps_text
)

print("GPS Coordinates:")
for lat, lat_dir, lon, lon_dir in coords:
    lat_val = float(lat) * (1 if lat_dir == "N" else -1)
    lon_val = float(lon) * (-1 if lon_dir == "W" else 1)
    print(f"  ({lat_val:.4f}, {lon_val:.4f})")

---

## Part 6: Real Parsing Examples

Let us apply everything to real geotechnical parsing tasks.

In [ ]:
# ─────────────────────────────────────────────────────
# EXAMPLE 1: Parse a complete borehole entry
# ─────────────────────────────────────────────────────

entry = "BH-01 @ 3.5m: N=24, Silty Sand (SM), w=18.5%"
print(f"Input: '{entry}'")
print()

# Extract each field with its own pattern
bh_id    = re.search(r"(BH-\d+)", entry)
depth    = re.search(r"(\d+\.\d+)m", entry)
n_value  = re.search(r"N=(\d+)", entry)
soil     = re.search(r"N=\d+,\s*(.+?)\s*\(", entry)
uscs     = re.search(r"\(([A-Z]{2})\)", entry)
moisture = re.search(r"w=(\d+\.?\d*)%", entry)

# Build a structured dictionary
parsed = {
    "bh_id":    bh_id.group(1)    if bh_id    else None,
    "depth_m":  float(depth.group(1))  if depth    else None,
    "N60":      int(n_value.group(1))  if n_value  else None,
    "soil":     soil.group(1)     if soil     else None,
    "uscs":     uscs.group(1)     if uscs     else None,
    "w_pct":    float(moisture.group(1)) if moisture else None,
}

print("Parsed output:")
for key, val in parsed.items():
    print(f"  {key:10s}: {val}")
print()
print("From one messy string -> clean, typed dictionary!")

In [ ]:
# ─────────────────────────────────────────────────────
# EXAMPLE 2: Parse USCS classifications from mixed text
# ─────────────────────────────────────────────────────

# Real lab report might mix USCS codes with other text
lab_report = """Sample 1: SM - Silty Sand, LL=28, PI=6
Sample 2: CL - Low plasticity clay, LL=42, PI=18
Sample 3: SP-SM - Poorly graded sand with silt
Sample 4: CH - Fat clay, LL=62, PI=35
Sample 5: GW-GM - Well graded gravel with silt"""

# Extract USCS codes (handle dual classifications like SP-SM)
uscs_pattern = r"([SGCMOP][WPLHCTM](?:-[SGCMOP][WPLHCTM])?)"
classifications = re.findall(uscs_pattern, lab_report)
print(f"USCS codes: {classifications}")

# Extract Atterberg limits where available
ll_values = re.findall(r"LL=(\d+)", lab_report)
pi_values = re.findall(r"PI=(\d+)", lab_report)
print(f"LL values: {ll_values}")
print(f"PI values: {pi_values}")
print()

# Extract full sample records
sample_pattern = r"Sample (\d+):\s+([A-Z]{2}(?:-[A-Z]{2})?)\s*-\s*(.+?)(?:,|$)"
samples = re.findall(sample_pattern, lab_report, re.MULTILINE)
print("Parsed samples:")
for num, uscs, desc in samples:
    print(f"  #{num}: {uscs:6s} | {desc}")

In [ ]:
# ─────────────────────────────────────────────────────
# EXAMPLE 3: Clean messy field notes
# ─────────────────────────────────────────────────────

# Field notes written by different drillers, inconsistent formats
messy_notes = [
    "Depth 3.5 meters - SPT blow count = 24  (silty SAND)",
    "at 6.0m, N-value: 12, CLAY  (low plasticity)",
    "9.0 m depth; N = 35;  GRAVEL, well graded",
    "depth=12.0m N=50 Weathered  ROCK",
]

print("Cleaning messy field notes:")
print("=" * 60)

for note in messy_notes:
    print(f"  Raw:    '{note}'")
    
    # Extract depth (handle: "3.5 meters", "6.0m", "9.0 m", "12.0m")
    depth_match = re.search(r"(\d+\.\d+)\s*(?:meters|m)", note)
    depth = float(depth_match.group(1)) if depth_match else None
    
    # Extract N-value (handle: "blow count = 24", "N-value: 12", "N = 35", "N=50")
    n_match = re.search(r"(?:N[- ]?value|N|blow count)\s*[=:]\s*(\d+)", note, re.IGNORECASE)
    n_val = int(n_match.group(1)) if n_match else None
    
    # Extract soil type (uppercase words: SAND, CLAY, GRAVEL, ROCK)
    soil_match = re.findall(r"\b([A-Z]{3,})\b", note)
    # Filter out common non-soil words
    non_soil = {"SPT", "GPS", "BH", "CPT"}
    soils = [s.title() for s in soil_match if s not in non_soil]
    soil = " ".join(soils) if soils else None
    
    print(f"  Parsed: depth={depth}m, N={n_val}, soil={soil}")
    print()

---

## Part 7: Complete Application — Borehole Log Parser

We will build a parser that takes raw, unstructured borehole log text and extracts structured data into a clean list of dictionaries — ready for analysis or conversion to a Pandas DataFrame.

In [ ]:
# ══════════════════════════════════════════════════════
# COMPLETE APPLICATION: BOREHOLE LOG PARSER
# Raw text -> Structured data using regex
# ══════════════════════════════════════════════════════

import re

def parse_borehole_log(raw_text):
    """Parse raw borehole log text into structured records.
    
    Handles multiple common formats:
      - BH-01 @ 3.5m: N=24, Silty Sand (SM), w=18.5%
      - BH-01 | 6.0 m | N-value=12 | Clay (CL) | moisture 32.1%
      - Borehole 01, depth 9.0m, SPT=35, Gravel
    
    Returns: list of dicts with keys:
      bh_id, depth_m, N60, soil, uscs, w_pct
    """
    records = []
    
    # Compile patterns once for efficiency
    pat_bh    = re.compile(r"(?:BH|Borehole)[- ]?(\d+)")
    pat_depth = re.compile(r"(\d+\.\d+)\s*m")
    pat_n     = re.compile(r"(?:N[- ]?(?:value)?|SPT)\s*[=:]?\s*(\d+)")
    pat_uscs  = re.compile(r"\(([A-Z]{2}(?:-[A-Z]{2})?)\)")
    pat_w     = re.compile(r"(?:w|moisture|mc)\s*[=:]?\s*(\d+\.?\d*)\s*%")
    
    # Soil type: words between N-value and USCS code, or after last comma
    pat_soil  = re.compile(r"N[=:]?\s*\d+[,|]?\s*([A-Za-z][A-Za-z ]+?)\s*(?:\(|,|$)")
    
    for line in raw_text.strip().split("\n"):
        line = line.strip()
        if not line:
            continue
        
        # Extract each field (None if not found)
        bh_m    = pat_bh.search(line)
        dep_m   = pat_depth.search(line)
        n_m     = pat_n.search(line)
        uscs_m  = pat_uscs.search(line)
        w_m     = pat_w.search(line)
        soil_m  = pat_soil.search(line)
        
        record = {
            "bh_id":   f"BH-{bh_m.group(1).zfill(2)}" if bh_m   else None,
            "depth_m": float(dep_m.group(1))           if dep_m  else None,
            "N60":     int(n_m.group(1))               if n_m    else None,
            "soil":    soil_m.group(1).strip()         if soil_m else None,
            "uscs":    uscs_m.group(1)                 if uscs_m else None,
            "w_pct":   float(w_m.group(1))             if w_m    else None,
        }
        
        # Only keep lines where we found at least depth and N-value
        if record["depth_m"] is not None and record["N60"] is not None:
            records.append(record)
    
    return records


print("Borehole log parser ready.")
print("Handles: BH IDs, depths, N-values, soil types, USCS codes, moisture.")

In [ ]:
# ──────────────────────────────────────────────────────
# RUN THE PARSER on messy, inconsistent borehole data
# ──────────────────────────────────────────────────────

# Simulate raw log text from different sources/drillers
raw_log = """BH-01 @ 1.5m: N=5, Fill
BH-01 @ 3.0m: N=8, Silty Sand (SM), w=14.2%
BH-01 @ 4.5m: N=12, Sand (SP), w=16.8%
BH-01 @ 6.0m: N=18, Clayey Sand (SC), w=22.3%
BH-01 @ 7.5m: N=28, Sand (SW), w=19.1%
BH-01 @ 9.0m: N=35, Gravel (GP), w=12.5%
BH-01 @ 10.5m: N=50, Weathered Rock, w=8.2%
BH-01 @ 12.0m: N=60, Bedrock
BH-02 @ 1.5m: N=3, Fill, w=20.1%
BH-02 @ 3.0m: N=6, Clay (CL), w=35.4%
BH-02 @ 4.5m: N=10, Silty Sand (SM), w=18.7%
BH-02 @ 6.0m: N=15, Clayey Sand (SC), w=24.5%
BH-02 @ 7.5m: N=22, Sand (SP), w=17.2%
BH-02 @ 9.0m: N=30, Gravel (GP), w=11.8%
BH-02 @ 10.5m: N=45, Weathered Rock, w=7.5%
BH-02 @ 12.0m: N=55, Bedrock"""

# Parse it
records = parse_borehole_log(raw_log)

# Display as a clean table
print(f"Parsed {len(records)} records from raw text")
print()
print(f"  {'BH':6s} {'Depth':>6s} {'N60':>4s} {'Soil':18s} {'USCS':>5s} {'w%':>6s}")
print("  " + "─" * 50)

for r in records:
    uscs_str = r['uscs'] if r['uscs'] else "--"
    w_str = f"{r['w_pct']:.1f}" if r['w_pct'] is not None else "--"
    soil_str = r['soil'] if r['soil'] else "--"
    print(f"  {r['bh_id']:6s} {r['depth_m']:5.1f}m {r['N60']:4d} "
          f"{soil_str:18s} {uscs_str:>5s} {w_str:>6s}")

In [ ]:
# ──────────────────────────────────────────────────────
# ANALYZE THE PARSED DATA — now it is structured!
# ──────────────────────────────────────────────────────

# Per-borehole summary
bh_ids = list(set(r["bh_id"] for r in records))
bh_ids.sort()

print("=" * 60)
print("  SITE INVESTIGATION SUMMARY (parsed from raw text)")
print("=" * 60)

for bh_id in bh_ids:
    bh_records = [r for r in records if r["bh_id"] == bh_id]
    n_vals = [r["N60"] for r in bh_records]
    w_vals = [r["w_pct"] for r in bh_records if r["w_pct"] is not None]
    uscs_codes = [r["uscs"] for r in bh_records if r["uscs"]]
    max_depth = max(r["depth_m"] for r in bh_records)
    
    print(f"\n  {bh_id}")
    print(f"  {'-' * 40}")
    print(f"  Layers:     {len(bh_records)}")
    print(f"  Max depth:  {max_depth:.1f}m")
    print(f"  N60 range:  {min(n_vals)} - {max(n_vals)}")
    print(f"  N60 avg:    {sum(n_vals)/len(n_vals):.1f}")
    if w_vals:
        print(f"  w% range:   {min(w_vals):.1f} - {max(w_vals):.1f}")
    print(f"  USCS found: {uscs_codes}")
    
    # Classify by consistency
    loose  = sum(1 for n in n_vals if n < 10)
    medium = sum(1 for n in n_vals if 10 <= n < 30)
    dense  = sum(1 for n in n_vals if n >= 30)
    print(f"  Loose/Med/Dense: {loose}/{medium}/{dense}")

print()
print("=" * 60)
print("  All data parsed from unstructured text using regex!")
print("  In Phase 2: pd.DataFrame(records) -> instant table.")
print("=" * 60)

---

## Practice Exercises

### Exercise 1: Extract depths and N-values
Given the string `"BH-05 @ 4.5m: N=15, Sandy Clay (SC), w=28.3%"`, use `re.search()` with groups to extract the borehole ID, depth, N-value, USCS code, and moisture content.

### Exercise 2: Parse GPS coordinates
Write a regex to extract latitude and longitude from: `"Site location: 40.7608N, 111.8910W, elevation 1320.5m ASL"`. Convert to signed decimal degrees (N=positive, W=negative).

### Exercise 3: Clean and standardize
Use `re.sub()` to standardize these depth formats into `"X.Xm"` form:
- `"3.5 meters"` -> `"3.5m"`
- `"6.0 m"` -> `"6.0m"`
- `"9 m"` -> `"9m"`
- `"depth of 12.0 metres"` -> `"depth of 12.0m"`

In [ ]:
# EXERCISE 1
# entry = "BH-05 @ 4.5m: N=15, Sandy Clay (SC), w=28.3%"
# bh = re.search(r"...", entry)
# depth = re.search(r"...", entry)
# ...

In [ ]:
# EXERCISE 2
# gps = "Site location: 40.7608N, 111.8910W, elevation 1320.5m ASL"
# m = re.search(r"...", gps)
# lat = ...
# lon = ...

In [ ]:
# EXERCISE 3
# texts = ["3.5 meters", "6.0 m", "9 m", "depth of 12.0 metres"]
# for t in texts:
#     cleaned = re.sub(r"...", r"...", t)
#     print(f"  '{t}' -> '{cleaned}'")

---

## Key Takeaways

1. **Regular expressions** solve the unstructured text problem — parsing borehole logs, field notes, lab reports, and GPS strings that are too messy for `split()` or `find()`.

2. **`re.search()`** finds the first match, **`re.findall()`** finds all matches, **`re.sub()`** does find-and-replace. Always check for `None` before calling `.group()`.

3. **Capturing groups** `()` extract specific values from within a pattern. Named groups `(?P<name>...)` convert matches directly to dictionaries via `.groupdict()`.

4. **Character classes** `[A-Z]{2}` let you precisely target USCS codes, borehole IDs, and coordinate formats without matching unrelated text.

5. **Compiled patterns** (`re.compile()`) are faster when parsing many lines — the regex engine only builds the pattern once, then reuses it.

6. **Always use raw strings** `r"..."` for regex patterns to avoid Python interpreting backslashes before the regex engine sees them.

---

## How This Connects to ML

| Today's Concept | ML Equivalent |
|----------------|---------------|
| `re.findall()` extraction | Feature extraction from text data |
| Parsing borehole logs | Data ingestion / ETL pipelines |
| `re.sub()` cleanup | Data cleaning & preprocessing |
| Named groups -> dict | Structured feature engineering |
| Character classes | NLP tokenization patterns |
| Compiled patterns | Efficient batch processing |
| Text -> structured data | Unstructured -> tabular conversion |
| Pattern validation | Input validation for ML pipelines |
| Coordinate parsing | Geospatial feature extraction |

In real ML projects, 60-80% of the work is **data cleaning**. Most geotechnical data starts as PDFs, scanned logs, or handwritten notes. Regex is the bridge between those raw sources and the clean DataFrames that ML models need.

---

*Day 014 of 100 — Regex turns messy text into structured data!*